In [1]:
# ============================================================
# Notebook : 11_household_uncertainty.ipynb
# Purpose  : Household-level description and uncertainty (review 3.4).
#   - Per-household table: sessions, observations, PCMCI-eligible sessions,
#     median session length, missing-step rate, overall abnormal rate.
#   - Session-length skew: share of PCMCI session-steps held by the largest
#     households (effective sample concentration).
#   - Household bootstrap (resample 22 PCMCI households with replacement,
#     B=1000): for each category, fraction of bootstrap samples whose lag-1
#     top precursor matches the full-sample top precursor, its sign-stability,
#     and a 95% percentile interval for the top-precursor partial correlation.
#   Partial correlations: pooled OLS, controlling for the category's AR term.
# Design   : Single code cell. Grayscale figures, 600 dpi, png+pdf. No path print.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.family": "serif", "axes.facecolor": "white",
                     "figure.facecolor": "white", "savefig.facecolor": "white",
                     "savefig.dpi": 600, "figure.dpi": 120,
                     "axes.edgecolor": "0.15", "grid.color": "0.85"})


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")


def partial_corrs(y, X):
    n = len(y)
    Xd = np.column_stack([np.ones(n), X])
    beta, _, _, _ = np.linalg.lstsq(Xd, y, rcond=None)
    resid = y - Xd @ beta
    dof = n - Xd.shape[1]
    s2 = (resid @ resid) / dof
    inv = np.linalg.pinv(Xd.T @ Xd)
    se = np.sqrt(np.maximum(np.diag(inv) * s2, 1e-18))
    tv = beta / se
    return (tv / np.sqrt(tv ** 2 + dof))[1:]


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
SENSORS = meta["sensor_cont"]
CATS = meta["abnormal_categories"]
MIN_LEN = meta["min_pcmci_len"]
CAT_KO = {"HR": "심박이상", "BR": "호흡이상", "ST": "체온이상",
          "SP": "산소포화도이상", "ER": "응급직접신호"}
df = df.sort_values(["session_id", "Timestamp"]).reset_index(drop=True)

# ----- Per-household summary -----
sess_len = df.groupby(["HH_ID", "session_id"]).size().rename("len").reset_index()
rows = []
for hh, g in df.groupby("HH_ID"):
    sl = sess_len[sess_len.HH_ID == hh]["len"]
    n_elig = int((sl >= MIN_LEN).sum())
    rows.append({"HH_ID": hh, "n_sessions": g["session_id"].nunique(),
                 "n_obs": len(g), "n_pcmci_sessions": n_elig,
                 "median_session_len": int(sl.median()),
                 "max_session_len": int(sl.max()),
                 "abnormal_rate": round(float(g["any_abnormal"].mean()), 4)})
hh_tbl = pd.DataFrame(rows).sort_values("n_obs", ascending=False).reset_index(drop=True)
hh_tbl.to_csv(TAB / "table_household_summary.csv", index=False, encoding="utf-8-sig")

# ----- Session-length concentration among PCMCI-eligible sessions -----
elig = sess_len[sess_len.len >= MIN_LEN].copy()
hh_steps = elig.groupby("HH_ID")["len"].sum().sort_values(ascending=False)
total_steps = hh_steps.sum()
share_top1 = round(float(hh_steps.iloc[0] / total_steps), 4)
share_top3 = round(float(hh_steps.iloc[:3].sum() / total_steps), 4)
n_pcmci_hh = int(hh_steps.shape[0])

# ----- Household bootstrap of the lag-1 top precursor -----
def top_precursor(frame, c):
    y = frame[f"cat_{c}"].astype(float).values
    ar = frame.groupby("session_id")[f"cat_{c}"].shift(1)
    lagS = frame.groupby("session_id")[SENSORS].shift(1)
    D = pd.concat([pd.Series(y, index=frame.index, name="y"), lagS,
                   ar.rename("ar")], axis=1).dropna()
    if D["y"].sum() < 15 or len(D) < 150:
        return None, None
    r = partial_corrs(D["y"].values, D[SENSORS + ["ar"]].values)[:len(SENSORS)]
    i = int(np.argmax(np.abs(r)))
    return SENSORS[i], float(r[i])

pcmci_hh = list(hh_steps.index)
base = {c: top_precursor(df[df.HH_ID.isin(pcmci_hh)], c) for c in CATS}

rng = np.random.default_rng(42)
B = 1000
boot_rows = []
for c in CATS:
    base_sensor, base_val = base[c]
    if base_sensor is None:
        boot_rows.append({"category": c, "category_ko": CAT_KO[c],
                          "base_top": "직접신호(원시)", "base_parcorr": np.nan,
                          "match_rate": np.nan, "sign_stability": np.nan,
                          "ci_low": np.nan, "ci_high": np.nan})
        continue
    matches, signs, vals = 0, 0, []
    for _ in range(B):
        samp = rng.choice(pcmci_hh, size=len(pcmci_hh), replace=True)
        sub = pd.concat([df[df.HH_ID == h] for h in samp], ignore_index=True)
        s, v = top_precursor(sub, c)
        if s is None:
            continue
        vals.append(v)
        if s == base_sensor:
            matches += 1
        # sign stability: sign of the base sensor's own partial corr
        y = sub[f"cat_{c}"].astype(float).values
        ar = sub.groupby("session_id")[f"cat_{c}"].shift(1)
        lagS = sub.groupby("session_id")[SENSORS].shift(1)
        D = pd.concat([pd.Series(y, index=sub.index, name="y"), lagS, ar.rename("ar")], axis=1).dropna()
        rr = partial_corrs(D["y"].values, D[SENSORS + ["ar"]].values)[:len(SENSORS)]
        if np.sign(rr[SENSORS.index(base_sensor)]) == np.sign(base_val):
            signs += 1
    vals = np.array(vals)
    boot_rows.append({"category": c, "category_ko": CAT_KO[c],
                      "base_top": base_sensor, "base_parcorr": round(base_val, 4),
                      "match_rate": round(matches / B, 3),
                      "sign_stability": round(signs / B, 3),
                      "ci_low": round(float(np.percentile(vals, 2.5)), 4),
                      "ci_high": round(float(np.percentile(vals, 97.5)), 4)})
boot = pd.DataFrame(boot_rows)
boot.to_csv(TAB / "table_household_bootstrap.csv", index=False, encoding="utf-8-sig")

# ----- Figure: bootstrap CI of top-precursor partial correlation -----
fig, ax = plt.subplots(figsize=(7.2, 4.0))
plot = boot.dropna(subset=["base_parcorr"])
yv = np.arange(len(plot))
ax.errorbar(plot["base_parcorr"], yv,
            xerr=[plot["base_parcorr"] - plot["ci_low"], plot["ci_high"] - plot["base_parcorr"]],
            fmt="o", color="0.15", ecolor="0.45", capsize=4, markersize=6)
ax.axvline(0, color="0.6", lw=1, ls="--")
ax.set_yticks(yv)
ax.set_yticklabels([f"{r.category} ({r.base_top})" for r in plot.itertuples()])
ax.set_xlabel("lag-1 partial correlation of top precursor (household bootstrap 95% CI)")
fig.tight_layout()
save_fig(fig, "fig_household_bootstrap_ci")
plt.close(fig)

print("=== Per-household summary (head) ===")
print(hh_tbl.head(8).to_string(index=False))
print(f"\nPCMCI-eligible households: {n_pcmci_hh}; "
      f"largest household step share={share_top1}, top-3 share={share_top3}")
print("\n=== Household bootstrap (B=1000) ===")
print(boot.to_string(index=False))
print("\nSaved: table_household_summary.csv, table_household_bootstrap.csv, fig_household_bootstrap_ci")


=== Per-household summary (head) ===
HH_ID  n_sessions  n_obs  n_pcmci_sessions  median_session_len  max_session_len  abnormal_rate
A0028          26   6397                10                  62             1785         0.1077
A0003          12   6299                 9                 396             2437         0.3037
A0022          12   5920                 8                 307             1217         0.2093
A0010          10   5905                 9                 395             1958         0.1876
A0004          10   5785                 7                 407             1940         0.2510
A0024          37   5622                10                  60             1029         0.2287
A0018          11   5406                10                 432             1400         0.2248
A0013          24   5347                 8                  42             2304         0.2138

PCMCI-eligible households: 22; largest household step share=0.0704, top-3 share=0.2054

=== Household boots